# Lab 2: Tool-Use Agent — Give Your Agent the Ability to Act

**Module 1: Foundations of Agentic AI**  
**Duration:** 30 minutes  
**Difficulty:** Intermediate  
**Prerequisite:** Lab 1 (Hello Agent)

---

## What You'll Learn

1. Define tools using Claude's tool-use JSON schema
2. Send requests that trigger autonomous tool selection
3. Handle `tool_use` responses and return `tool_result` messages
4. Watch the agent reason → select tool → act → observe in real time
5. Understand why tool design matters more than prompt engineering

## How This Connects to the Agent Loop

```mermaid
flowchart TD
    A["PERCEIVE<br>User sends a task"] --> B["REASON<br>Claude decides WHICH TOOL to use"]
    B --> C["ACT<br>Your code executes the tool"]
    C --> D["OBSERVE<br>Claude sees the result"]
    style A fill:#bee3f8,color:#1a202c
    style B fill:#c4b5fd,color:#1a202c,stroke:#7c3aed,stroke-width:3px
    style C fill:#fed7aa,color:#1a202c,stroke:#dd6b20,stroke-width:3px
    style D fill:#bbf7d0,color:#1a202c,stroke:#38a169,stroke-width:3px
```

**The big idea:** In Lab 1, Claude could only *talk*. Now it can *do things*.

---
## Step 1: Setup

In [ ]:
!pip install -q anthropic

In [ ]:
import os
import json
import math
from getpass import getpass
from datetime import datetime

import anthropic

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("Enter your Anthropic API key: ")

client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-5-20241022"

print(f"✓ Ready. Model: {MODEL}")

---
## Step 2: Define Your Tools

Tools are defined as JSON schemas that tell Claude:
1. **What** the tool does (name + description)
2. **What inputs** it needs (parameters with types and descriptions)

Claude reads these descriptions to decide which tool to use. **Good descriptions = good tool selection.** This is why tool design matters more than prompt engineering.

Let's create three tools:

In [ ]:
# Define the tools Claude can use
tools = [
    {
        "name": "calculator",
        "description": (
            "Perform mathematical calculations. Supports basic arithmetic "
            "(+, -, *, /), exponents (**), square roots (sqrt), and common "
            "math functions. Use this whenever you need to compute a number."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "The math expression to evaluate, e.g. '(15 * 24) + 7'"
                }
            },
            "required": ["expression"]
        }
    },
    {
        "name": "get_current_time",
        "description": (
            "Get the current date and time. Use this when the user asks about "
            "today's date, current time, or when you need to timestamp something."
        ),
        "input_schema": {
            "type": "object",
            "properties": {},
            "required": []
        }
    },
    {
        "name": "word_counter",
        "description": (
            "Count the number of words, characters, and sentences in a given text. "
            "Use this for text analysis tasks."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "text": {
                    "type": "string",
                    "description": "The text to analyze"
                }
            },
            "required": ["text"]
        }
    }
]

print(f"✓ Defined {len(tools)} tools: {[t['name'] for t in tools]}")

## Step 3: Implement the Tool Functions

Each tool schema needs a corresponding Python function that actually does the work. The agent decides WHICH tool to call; your code EXECUTES it.

In [ ]:
def calculator(expression: str) -> str:
    """Safely evaluate a math expression.
    
    Args:
        expression: A math expression string like '(15 * 24) + 7'.
    
    Returns:
        The result as a string, or an error message.
    """
    # Allow only safe math operations
    allowed_names = {
        "sqrt": math.sqrt, "abs": abs, "round": round,
        "min": min, "max": max, "pow": pow,
        "pi": math.pi, "e": math.e,
        "sin": math.sin, "cos": math.cos, "tan": math.tan,
        "log": math.log, "log10": math.log10,
    }
    try:
        result = eval(expression, {"__builtins__": {}}, allowed_names)
        return str(result)
    except Exception as e:
        return f"Error: {e}"


def get_current_time() -> str:
    """Get the current date and time.
    
    Returns:
        Formatted date/time string.
    """
    now = datetime.now()
    return now.strftime("%Y-%m-%d %H:%M:%S (%A)")


def word_counter(text: str) -> str:
    """Count words, characters, and sentences in text.
    
    Args:
        text: The text to analyze.
    
    Returns:
        JSON string with word, character, and sentence counts.
    """
    words = len(text.split())
    characters = len(text)
    sentences = text.count('.') + text.count('!') + text.count('?')
    return json.dumps({
        "words": words,
        "characters": characters,
        "sentences": max(sentences, 1)
    })


# Map tool names to functions for easy dispatch
TOOL_FUNCTIONS = {
    "calculator": calculator,
    "get_current_time": get_current_time,
    "word_counter": word_counter,
}

print("✓ Tool functions implemented")

---
## Step 4: Make a Tool-Use Request

Now let's send Claude a task that requires a tool. Watch how Claude **decides** to use a tool instead of guessing the answer.

In [ ]:
response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    tools=tools,
    messages=[{
        "role": "user",
        "content": "What is 1847 * 293 + 17?"
    }]
)

print(f"Stop reason: {response.stop_reason}")
print(f"Number of content blocks: {len(response.content)}")
print()

for block in response.content:
    if block.type == "text":
        print(f"[TEXT] {block.text}")
    elif block.type == "tool_use":
        print(f"[TOOL_USE] Tool: {block.name}")
        print(f"           ID:   {block.id}")
        print(f"           Input: {json.dumps(block.input)}")

**What just happened:**
1. Claude received the math question and the list of available tools
2. Claude **reasoned** that this needs calculation, not guessing
3. Claude returned a `tool_use` content block with the tool name and input
4. The `stop_reason` is `"tool_use"` — Claude is pausing, waiting for the result

**Claude did NOT compute the answer. It asked YOU to run the calculator.** This is the tool-use pattern.

---
## Step 5: Execute the Tool and Return the Result

Now we complete the loop: execute the tool, then send the result back to Claude so it can formulate its final answer.

In [ ]:
def process_tool_call(response) -> tuple[str, str, str]:
    """Extract tool call details from a response.
    
    Args:
        response: The Claude API response containing a tool_use block.
    
    Returns:
        Tuple of (tool_name, tool_input, tool_use_id).
    """
    for block in response.content:
        if block.type == "tool_use":
            return block.name, block.input, block.id
    raise ValueError("No tool_use block found in response")


def execute_tool(tool_name: str, tool_input: dict) -> str:
    """Execute a tool function and return the result.
    
    Args:
        tool_name: Name of the tool to execute.
        tool_input: Dictionary of input parameters.
    
    Returns:
        The tool's output as a string.
    """
    func = TOOL_FUNCTIONS.get(tool_name)
    if not func:
        return f"Error: Unknown tool '{tool_name}'"
    try:
        return func(**tool_input)
    except Exception as e:
        return f"Error executing {tool_name}: {e}"

In [ ]:
# Step 1: Extract the tool call
tool_name, tool_input, tool_use_id = process_tool_call(response)
print(f"Agent wants to call: {tool_name}({tool_input})")

# Step 2: Execute the tool
tool_result = execute_tool(tool_name, tool_input)
print(f"Tool result: {tool_result}")

# Step 3: Send the result back to Claude
final_response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    tools=tools,
    messages=[
        # Original user message
        {"role": "user", "content": "What is 1847 * 293 + 17?"},
        # Claude's response (including the tool_use block)
        {"role": "assistant", "content": response.content},
        # The tool result
        {
            "role": "user",
            "content": [{
                "type": "tool_result",
                "tool_use_id": tool_use_id,
                "content": tool_result
            }]
        }
    ]
)

print(f"\nFinal answer: {final_response.content[0].text}")
print(f"Stop reason: {final_response.stop_reason}")

**The complete flow:**
```mermaid
sequenceDiagram
    participant U as User
    participant C as Claude
    participant T as Your Code
    U->>C: "What is 1847 * 293 + 17?"
    Note over C: REASONS: I need the calculator
    C-->>T: tool_use(calculator, "1847*293+17")
    T->>T: calculator("1847*293+17") = 541388
    T-->>C: tool_result: "541388"
    Note over C: OBSERVES the result
    C-->>U: "The answer is 541,388"
```

---
## Step 6: Autonomous Tool Selection

The real power of tool-use is that **the agent decides which tool to use**. Let's give Claude different tasks and watch it choose the right tool each time.

In [ ]:
def ask_agent(question: str) -> str:
    """Send a question to the agent, handle any tool calls, return final answer.
    
    Args:
        question: The user's question.
    
    Returns:
        The agent's final response after tool execution.
    """
    print(f"\n{'='*60}")
    print(f"User: {question}")
    print(f"{'='*60}")
    
    messages = [{"role": "user", "content": question}]
    
    response = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        tools=tools,
        messages=messages
    )
    
    # If no tool was used, return the text directly
    if response.stop_reason == "end_turn":
        print(f"[No tool needed]")
        return response.content[0].text
    
    # Handle tool use
    tool_name, tool_input, tool_use_id = process_tool_call(response)
    print(f"[Agent chose: {tool_name}] Input: {tool_input}")
    
    tool_result = execute_tool(tool_name, tool_input)
    print(f"[Tool result: {tool_result}]")
    
    # Send result back for final answer
    messages.append({"role": "assistant", "content": response.content})
    messages.append({
        "role": "user",
        "content": [{"type": "tool_result", "tool_use_id": tool_use_id, "content": tool_result}]
    })
    
    final = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        tools=tools,
        messages=messages
    )
    
    answer = final.content[0].text
    print(f"\nAgent: {answer}")
    return answer

In [ ]:
# Test 1: Should use calculator
ask_agent("If I invest $10,000 at 7% annual compound interest, how much will I have after 5 years?")

# Test 2: Should use get_current_time
ask_agent("What day of the week is it today?")

# Test 3: Should use word_counter
ask_agent("How many words are in: 'The quick brown fox jumps over the lazy dog near the river bank'")

# Test 4: Should NOT use any tool
ask_agent("What is the capital of France?")

**Key observation:** Claude autonomously selected the right tool for each task — and knew when NO tool was needed (Test 4). This is **agentic behavior**: the model makes decisions about its own actions.

---
## Step 7: Handling Errors Gracefully

Real-world tools fail. A production agent needs to handle errors and recover. Let's see what happens when a tool returns an error.

In [ ]:
# Force a calculator error
result = calculator("1 / 0")
print(f"Calculator error: {result}")

# The agent should handle this gracefully
ask_agent("What is 100 divided by 0?")

**Notice:** Claude receives the error message from the tool and explains the problem to the user instead of crashing. This error-handling flow is essential for production agents.

---
## Step 8: Building a Reusable Tool-Use Agent

Let's wrap everything into a clean, reusable class that handles the full tool-use cycle.

In [ ]:
class ToolAgent:
    """An agent that can autonomously select and use tools to complete tasks."""
    
    def __init__(
        self,
        tools: list[dict],
        tool_functions: dict,
        system_prompt: str = "You are a helpful assistant with access to tools. Use them when needed.",
        model: str = MODEL,
    ):
        """Initialize the tool-use agent.
        
        Args:
            tools: List of tool definitions (JSON schemas).
            tool_functions: Dict mapping tool names to Python callables.
            system_prompt: System prompt defining agent behavior.
            model: The Claude model to use.
        """
        self.client = anthropic.Anthropic()
        self.model = model
        self.tools = tools
        self.tool_functions = tool_functions
        self.system_prompt = system_prompt
    
    def run(self, user_message: str, verbose: bool = True) -> str:
        """Send a task to the agent. Handles one round of tool use.
        
        Args:
            user_message: The user's request.
            verbose: If True, print the tool selection process.
        
        Returns:
            The agent's final response.
        """
        messages = [{"role": "user", "content": user_message}]
        
        response = self.client.messages.create(
            model=self.model,
            max_tokens=1024,
            system=self.system_prompt,
            tools=self.tools,
            messages=messages
        )
        
        # No tool needed — return text directly
        if response.stop_reason == "end_turn":
            return response.content[0].text
        
        # Handle tool use
        tool_name, tool_input, tool_use_id = None, None, None
        for block in response.content:
            if block.type == "tool_use":
                tool_name = block.name
                tool_input = block.input
                tool_use_id = block.id
                break
        
        if verbose:
            print(f"  🔧 Using tool: {tool_name}({json.dumps(tool_input)})")
        
        # Execute the tool
        func = self.tool_functions.get(tool_name)
        if func:
            try:
                tool_result = func(**tool_input)
            except Exception as e:
                tool_result = f"Error: {e}"
        else:
            tool_result = f"Error: Unknown tool '{tool_name}'"
        
        if verbose:
            print(f"  📋 Result: {tool_result}")
        
        # Return result to Claude
        messages.append({"role": "assistant", "content": response.content})
        messages.append({
            "role": "user",
            "content": [{"type": "tool_result", "tool_use_id": tool_use_id, "content": tool_result}]
        })
        
        final = self.client.messages.create(
            model=self.model,
            max_tokens=1024,
            system=self.system_prompt,
            tools=self.tools,
            messages=messages
        )
        
        return final.content[0].text


# Create the agent
agent = ToolAgent(
    tools=tools,
    tool_functions=TOOL_FUNCTIONS,
    system_prompt=(
        "You are a precise assistant. When you need to compute something, "
        "always use the calculator — never do mental math. "
        "When asked about the current date/time, use the time tool."
    )
)

# Test it
print(agent.run("How many seconds are in a year? (365 days)"))

---
## Exercises

### Exercise 1: Add a New Tool

Add a `string_reverser` tool that reverses any given text. You need to:
1. Define the tool schema (JSON)
2. Write the Python function
3. Add it to the tools list and TOOL_FUNCTIONS dict
4. Test it with the agent

In [ ]:
# YOUR CODE HERE
# 1. Define the tool schema
# 2. Write the function
# 3. Add to tools list and TOOL_FUNCTIONS
# 4. Test with: agent.run("Reverse the text: 'Hello World'")



### Exercise 2: Tool Description Experiment

Change the `calculator` tool's description to something vague like `"A math tool"`. Then test it with the same questions. Does Claude still choose it correctly? Now try a misleading description. What happens?

**This demonstrates why tool design > prompt engineering.**

In [ ]:
# YOUR CODE HERE
# Experiment with different tool descriptions and observe the effect



### Exercise 3: Multi-Tool Task

Ask the agent a question that logically requires TWO tools (e.g., "What's today's date, and how many days until December 31st?"). What happens with our current single-tool-call implementation? How would you fix it?

*(Hint: you'll solve this properly in Lab 3 with the agent loop)*

In [ ]:
# YOUR CODE HERE
# Try a multi-tool question and observe what happens



---
## Key Takeaways

1. **Tools transform a chatbot into an agent** — they let the LLM interact with the real world
2. **Claude decides which tool to use** — you define the tools, Claude selects autonomously
3. **The tool-use protocol** is: `tool_use` response → execute tool → send `tool_result` → get final answer
4. **Tool descriptions are critical** — clear descriptions lead to correct tool selection
5. **Error handling is essential** — tools fail in production; agents must recover gracefully

## Limitation of This Lab

Our agent handles only **one** tool call per request. What if a task requires multiple tools in sequence? That's exactly what Lab 3 solves with the **agent loop**.

## What's Next

In **Lab 3**, we'll build a **multi-step agent** that:
- Loops until the task is complete
- Uses multiple tools in sequence
- Maintains conversation memory across tool calls
- Decides when it's done

---
*Lab 2 Complete — Module 1: Foundations of Agentic AI*